In [1]:
import overpy
import time
import pandas as pd


#api = overpy.Overpass()
api = overpy.Overpass(url="https://overpass.kumi.systems/api/interpreter")

In [2]:
# Consulta Overpass ampliada para bancos e comércios em geral
# A hierarquia administrativa (admin_level):
# 2 = país (Brasil)
# 4 = estado (ex.: São Paulo)
# 6 = município
# 8 = distrito/subprefeitura
# 10 = bairro

query = """
[out:json][timeout:1000];
area["boundary"="administrative"]["name"="Brasil"]["admin_level"="2"]->.searchArea;

(
    node["shop"](area.searchArea);
    way["shop"](area.searchArea);
    node["amenity"="bank"](area.searchArea);
    way["amenity"="bank"](area.searchArea);
    node["office"="bank"](area.searchArea);
    way["office"="bank"](area.searchArea);
);

out center;
"""

# Função que faz a consulta com tentativa e erro + pausa
def safe_query(api, query, max_tries=5, wait_seconds=30):
    for attempt in range(1, max_tries + 1):
        try:
            print(f"Tentando consulta (tentativa {attempt}/{max_tries})...")
            result = api.query(query)
            print("Consulta concluída com sucesso!")
            return result
        except overpy.exception.OverpassTooManyRequests:
            print(f"Muitas requisições — aguardando {wait_seconds}s antes de tentar novamente...")
            time.sleep(wait_seconds)
        except overpy.exception.OverpassGatewayTimeout:
            print("Tempo excedido pelo servidor — aguardando um pouco e tentando de novo...")
            time.sleep(wait_seconds)
        except Exception as e:
            print(f"Erro inesperado: {e}")
            time.sleep(wait_seconds)
    raise RuntimeError("Limite de tentativas atingido sem sucesso")

result = safe_query(api, query)

Tentando consulta (tentativa 1/5)...
Consulta concluída com sucesso!


In [3]:
# Transformar os resultados em DataFrame
data = []

# Nodes
for node in result.nodes:
    data.append({
        "name": node.tags.get("name", "Sem nome"),
        "tipo": node.tags.get("shop") or node.tags.get("amenity") or node.tags.get("office"),
        "latitude": node.lat,
        "longitude": node.lon,
        "cep": node.tags.get("addr:postcode", "Sem CEP")
    })

# Ways
for way in result.ways:
    if hasattr(way, "center_lat") and hasattr(way, "center_lon"):
        data.append({
            "name": way.tags.get("name", "Sem nome"),
            "tipo": way.tags.get("shop") or way.tags.get("amenity") or way.tags.get("office"),
            "latitude": way.center_lat,
            "longitude": way.center_lon,
            "cep": way.tags.get("addr:postcode", "Sem CEP")
        })

# Relations
for rel in result.relations:
    if hasattr(rel, "center_lat") and hasattr(rel, "center_lon"):
        data.append({
            "name": rel.tags.get("name", "Sem nome"),
            "tipo": rel.tags.get("shop") or rel.tags.get("amenity") or rel.tags.get("office"),
            "latitude": rel.center_lat,
            "longitude": rel.center_lon,
            "cep": rel.tags.get("addr:postcode", "Sem CEP")
        })

df = pd.DataFrame(data)
print("Total de estabelecimentos encontrados:", len(df))
df.head()

Total de estabelecimentos encontrados: 182576


,name,tipo,latitude,longitude,cep
0,Silvio Romero Plaza,mall,-23.5472892,-46.5727729,Sem CEP
1,Auto Shopping Cidade Morumbi,car,-23.6290122,-46.7172027,Sem CEP
2,Banco do Brasil,bank,-1.0555154,-46.7651224,Sem CEP
3,State Bank of Para,bank,-1.0549849,-46.7653066,Sem CEP
4,Banco da Amazônia,bank,-1.0567423,-46.7649376,Sem CEP


In [4]:
# Salvar como arquivo
df.to_csv("../../Data/Raw/centros_comerciais_brasil.csv", index=False, encoding="utf-8")
print("Arquivo salvo: centros_comerciais_brasil.csv")

Arquivo salvo: centros_comerciais_brasil.csv


In [5]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

# Carrega o dataset de centros comerciais existentes
df = pd.read_csv("../../Data/Raw/centros_comerciais_brasil.csv")

print("Dataset carregado com", len(df), "centros comerciais")
df.head()

Dataset carregado com 182576 centros comerciais


,name,tipo,latitude,longitude,cep
0,Silvio Romero Plaza,mall,-23.547289,-46.572773,Sem CEP
1,Auto Shopping Cidade Morumbi,car,-23.629012,-46.717203,Sem CEP
2,Banco do Brasil,bank,-1.055515,-46.765122,Sem CEP
3,State Bank of Para,bank,-1.054985,-46.765307,Sem CEP
4,Banco da Amazônia,bank,-1.056742,-46.764938,Sem CEP


In [6]:
import plotly.express as px

def criar_mapa_calor(df, lat_col="latitude", lon_col="longitude"):
    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()
    centro = {"lat": (lat_min + lat_max) / 2, "lon": (lon_min + lon_max) / 2}

    max_dist = max(lat_max - lat_min, lon_max - lon_min)
    if max_dist > 10: zoom = 4
    elif max_dist > 5: zoom = 6
    elif max_dist > 2: zoom = 8
    else: zoom = 10

    fig = px.density_mapbox(
        df,
        lat=lat_col,
        lon=lon_col,
        radius=30,
        center=centro,
        zoom=zoom,
        mapbox_style="carto-darkmatter",
        color_continuous_scale=[
            (0.0, "rgba(255, 255, 255, 0)"),
            (0.1, "rgba(255, 180, 180, 0.3)"),
            (0.3, "rgba(255, 100, 100, 0.5)"),
            (0.6, "rgba(255, 50, 50, 0.7)"),
            (1.0, "rgba(255, 0, 0, 1.0)")
        ],
        title="Mapa de Calor — Concentração de Estabelecimentos Comerciais"
    )

    fig.add_scattermapbox(
        lat=df[lat_col],
        lon=df[lon_col],
        mode="markers",
        marker=dict(size=4, color="rgba(255,255,255,0.6)"),
        name="Pontos Individuais"
    )

    fig.add_scattermapbox(
        lat=[centro["lat"]],
        lon=[centro["lon"]],
        mode="markers",
        marker=dict(size=10, color="cyan", symbol="cross"),
        name="Centro da Região"
    )

    fig.update_layout(
        height=800,
        margin=dict(r=0, t=60, l=0, b=0),
        title=dict(font=dict(size=22, color="#fafafa", family="Arial Black")),
        paper_bgcolor="#111111",
        font=dict(color="#eeeeee", family="Arial"),
        coloraxis_showscale=False,
        mapbox=dict(pitch=45, style="carto-darkmatter"),
        legend=dict(bgcolor="rgba(0,0,0,0.3)", font=dict(color="white"))
    )

    return fig

fig = criar_mapa_calor(
    df.sample(min(len(df), 5000), random_state=42),
    lat_col="latitude",
    lon_col="longitude"
)
fig.show()

C:\Users\henrique.mourao\AppData\Local\Temp\ipykernel_31068\3914005960.py:14: DeprecationWarning: *density_mapbox* is deprecated! Use *density_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.density_mapbox(


In [7]:
# Converter as coordenadas para array numérico
points = np.array(list(zip(df.latitude, df.longitude)))

# Criar estrutura de busca espacial
tree = cKDTree(points)

In [8]:
def potencial_de_loja(lat, lon, raio_km=50, peso_dist=1.5, peso_dens=1.0):
    """
    Calcula o potencial de abertura de loja em (lat, lon)
    baseado na distância média e densidade local de centros comerciais.
    Quanto maior o score, maior o potencial.
    """
    # Conversão: 1 grau ≈ 111 km
    raio_deg = raio_km / 111.0
    
    # Encontrar vizinhos dentro do raio
    ids = tree.query_ball_point([lat, lon], raio_deg)
    n_vizinhos = len(ids)
    
    # Calcular distância média até os vizinhos
    if n_vizinhos > 0:
        distancias = np.linalg.norm(tree.data[ids] - [lat, lon], axis=1)
        dist_media = np.mean(distancias)
    else:
        dist_media = raio_deg  # nenhum vizinho próximo
    
    # Regra: quanto mais longe dos shoppings existentes e menos vizinhos, maior o score
    score = (peso_dist * dist_media) - (peso_dens * n_vizinhos)
    return score

In [9]:
scores = []
for i, row in df.iterrows():
    score = potencial_de_loja(row.latitude, row.longitude)
    scores.append(score)

df["score"] = scores

print("🔢 Média de score no Brasil:", df["score"].mean())
df.head()

🔢 Média de score no Brasil: -4183.1375750600155


,name,tipo,latitude,longitude,cep,score
0,Silvio Romero Plaza,mall,-23.547289,-46.572773,Sem CEP,-19424.800172
1,Auto Shopping Cidade Morumbi,car,-23.629012,-46.717203,Sem CEP,-18985.771067
2,Banco do Brasil,bank,-1.055515,-46.765122,Sem CEP,-172.464303
3,State Bank of Para,bank,-1.054985,-46.765307,Sem CEP,-172.464361
4,Banco da Amazônia,bank,-1.056742,-46.764938,Sem CEP,-172.464392


In [10]:
import plotly.express as px

def criar_mapa_calor(df, lat_col="latitude", lon_col="longitude"):
    """
    Cria um mapa de calor interativo centrado automaticamente
    pela extensão das coordenadas
    """
    # Calcular limites geográficos
    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()
    
    # Calcular centro e zoom dinâmico
    centro = dict(lat=(lat_min + lat_max)/2, lon=(lon_min + lon_max)/2)
    
    # Zoom aproximado — quanto menor a extensão, mais próximo
    max_dist = max(lat_max - lat_min, lon_max - lon_min)
    if max_dist > 10:
        zoom = 4       # Brasil inteiro
    elif max_dist > 5:
        zoom = 6       # Estado
    elif max_dist > 2:
        zoom = 8       # Região
    else:
        zoom = 10      # Cidade / bairro

    # Criar o mapa de calor interativo
    fig = px.density_mapbox(
        df,
        lat=lat_col,
        lon=lon_col,
        radius=15,                # controla dispersão dos pontos
        center=centro,
        zoom=zoom,
        mapbox_style="carto-positron",  # fundo neutro
        color_continuous_scale="Reds",  # tons de vermelho
        title="Mapa de Calor — Concentração de Coordenadas"
    )

    # Ajustes visuais
    fig.update_layout(
        height=800,
        margin=dict(r=0, t=50, l=0, b=0),
        coloraxis_showscale=False      # oculta barra de intensidade (opcional)
    )

    return fig

In [11]:
top_potenciais = df.sort_values(by="score", ascending=False).head(20)
top_potenciais[["name", "latitude", "longitude", "score"]]

,name,latitude,longitude,score
172695,Supermercado Nova Santa Rosa,-8.281522,-44.567219,-1.0
24754,supermercado Progresso,-10.034638,-44.302912,-1.0
96384,Shopping Amazonas,-3.755195,-47.498774,-1.0
173153,Rio Jeans,2.305872,-51.635661,-1.0
92339,Sem nome,3.043581,-51.461088,-1.0
5257,Sem nome,-4.180566,-60.807105,-1.0
92276,Sem nome,-3.697738,-52.447799,-1.0
43318,Padaria e Mercearia Vilhena,-9.938701,-67.015708,-1.0
173730,Elias da Lanternagem,-1.767176,-46.542903,-1.0
181673,Sem nome,-5.815579,-61.299173,-1.0
